# 🤖 Notebook 04 — Fine-tuning DistilBERT

Same task as notebook 02, completely different model class: a **pretrained Transformer encoder** instead of bag-of-words.

| decision | value | why |
|---|---|---|
| model | `distilbert-base-uncased` (66M) | fits 4 GB VRAM; ~97% of bert-base quality, 2x faster |
| head | `Linear(768, 6)` + sigmoid | 6 **independent** labels -> BCE loss, not softmax (EDA: 61% of labeled comments carry 2-3 labels) |
| `max_len` | 192 tokens | attention memory grows as `heads x N x N`; covers the vast majority of comments (EDA: median well under 50 words) |
| batch / lr / epochs | 16 / 2e-5 / 1 | standard fine-tuning recipe; small LR protects the 66M pretrained weights |
| precision | fp16 autocast | halves activation memory + speeds up the RTX 3050 |

What is new compared to notebook 02:
- We start from weights that already encode English (6 books + Wikipedia pre-training) and only *adapt* them.
- The text pipeline changes completely: no TF-IDF, no cleaning hacks - the model consumes raw subword tokens.
- Training runs on your GPU (~20-40 min for 1 epoch). Every ~500 steps prints loss/steps/s/ETA.

Split is the **same seed-42 80/20** as the baseline so per-class AUCs are directly comparable.

## 1. Setup: device, seeds, sanity checks

In [ ]:
import gc
import random
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sns.set_theme(style='whitegrid', context='talk')

# Reproducibility: seed everything that touches randomness
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | {props.total_memory / 1024**3:.1f} GB VRAM')
else:
    raise RuntimeError(
        'No CUDA GPU found. Fine-tuning 159k comments on CPU would take many hours.\n'
        'Either run this notebook on Kaggle (Settings -> Accelerator -> GPU T4) '
        'or check that the CUDA build of torch is installed.')

In [ ]:
def find_project_root(marker='data'):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    raise FileNotFoundError(f"No '{marker}/' found above {Path.cwd()}")

ROOT = find_project_root()
DATA_DIR, SUB_DIR, FIG_DIR = ROOT / 'data', ROOT / 'submissions', ROOT / 'figures'
SUB_DIR.mkdir(exist_ok=True)

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
PRETTY = {'toxic': 'Toxic', 'severe_toxic': 'Severe toxic', 'obscene': 'Obscene',
          'threat': 'Threat', 'insult': 'Insult', 'identity_hate': 'Identity hate'}
C_BASE, C_BERT = '#4C72B0', '#C44E52'

# Baseline val AUCs from notebook 02 - hardcoded here so the final comparison
# chart works even if that notebook's variables are not in this kernel.
BASELINE_VAL_AUC = {'toxic': 0.9734, 'severe_toxic': 0.9830, 'obscene': 0.9865,
                    'threat': 0.9907, 'insult': 0.9802, 'identity_hate': 0.9733}

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
print(f'train: {train.shape}   test: {test.shape}')

In [ ]:
# Same split as notebook 02 (same seed) -> directly comparable validation numbers.
Y = train[LABELS].values
X_tr, X_val, Y_tr, Y_val = train_test_split(
    train['comment_text'].fillna('').astype(str).values, Y,
    test_size=0.2, random_state=SEED)
print(f'train: {len(X_tr):,}   val: {len(X_val):,}')

# Note: NO regex cleaning this time. The subword tokenizer absorbs URLs and IPs
# into mostly-harmless pieces; aggressive hand-cleaning is a classical-MLP-era habit.

## 2. Tokenization: WordPiece subwords

DistilBERT uses WordPiece (30,522 tokens). Rare words split into frequent pieces, so there are no out-of-vocabulary words - watch what it does to the all-caps attack from our EDA:

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 192
BATCH = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('vocab size:', tokenizer.vocab_size)

demo = 'COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK'
print('uncased  ->', tokenizer.tokenize(demo.lower()))
print('as-caps  ->', tokenizer.tokenize(demo))

# ...and what happens to an anonymous editor's IP address:
print('ip       ->', tokenizer.tokenize('hello 162.54.3.4 you vandal'))

In [ ]:
# Tokenize once, keep variable-length lists; padding happens per-batch in the
# collate function (dynamic padding = only pad to the longest comment IN the batch).
def tokenize(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LEN)

t0 = time.time()
enc_tr = tokenize(X_tr)
enc_val = tokenize(X_val)
enc_test = tokenize(test['comment_text'].fillna('').astype(str).values)
print(f'tokenized 313k comments in {time.time() - t0:.1f}s')

lens = [len(ids) for ids in enc_tr['input_ids']]
print(f'train token lengths: mean {np.mean(lens):.0f}, '
      f'p95 {int(np.percentile(lens, 95))}, max kept {max(lens)}')

In [ ]:
class CommentsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: self.encodings[k][i] for k in ('input_ids', 'attention_mask')}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.float)
        return item

def collate(batch):
    # dynamic padding: pad THIS batch to ITS longest sequence, not always to 192
    enc = tokenizer.pad(
        {'input_ids': [b['input_ids'] for b in batch],
         'attention_mask': [b['attention_mask'] for b in batch]},
        padding=True, return_tensors='pt')
    enc['labels'] = torch.stack([b['labels'] for b in batch])
    return enc

train_ds = CommentsDataset(enc_tr, Y_tr)
val_ds = CommentsDataset(enc_val, Y_val)
test_ds = CommentsDataset(enc_test, np.zeros((len(test), len(LABELS)), dtype=np.float32))
print(f'batches per epoch: {len(train_ds) // BATCH:,}')

## 3. The model: pretrained encoder + fresh 6-label head

`AutoModelForSequenceClassification` loads the pretrained 6-layer encoder and attaches a **new, randomly initialized** `Linear(768, 6)` classifier. With `problem_type='multi_label_classification'`, passing float labels to `forward()` computes **BCEWithLogitsLoss** (independent sigmoid per label) - exactly our multi-label setup. The pre-training heads (MLM / NSP) are gone.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), problem_type='multi_label_classification')
model.to(device)

total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.classifier.parameters())
print(f'parameters: {total:,} total | only {head:,} start fresh (the classification head)')
print(f'trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS = 1
total_steps = len(train_loader) * EPOCHS
# linear warmup (post-LN stability!) then linear decay to 0
scheduler = get_scheduler('linear', optimizer=optimizer,
                          num_warmup_steps=500, num_training_steps=total_steps)
print(f'{total_steps:,} optimizer steps | warmup 500 | effective batch {BATCH}')

## 4. Train (1 epoch, fp16)

Memory math for our 4 GB card: fp16 weights ~130 MB + AdamW states ~520 MB + activations (attention is `12 heads x N x N`, N <= 192, batch 16) -> fits. `autocast` runs matmuls in fp16 while keeping the loss numerically safe; `GradScaler` avoids fp16 underflow in gradients.

In [ ]:
scaler = torch.amp.GradScaler('cuda')
model.train()
step, run_loss = 0, 0.0
t_start = time.time()

for epoch in range(EPOCHS):
    for batch in train_loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(**batch)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        run_loss += loss.item()
        step += 1
        if step % 500 == 0:
            elapsed = time.time() - t_start
            eta_min = (total_steps - step) / (step / elapsed) / 60
            print(f'step {step:6d}/{total_steps} | loss {run_loss / 500:.4f} | '
                  f'{step / elapsed:.2f} steps/s | lr {scheduler.get_last_lr()[0]:.2e} | ETA {eta_min:.0f} min')
            run_loss = 0.0

print(f'\ntraining done in {(time.time() - t_start) / 60:.1f} min')

## 5. Validate: per-class AUC vs the baseline

Watch for the same pattern as figure 08 - and remember `threat`'s val support is only 74: differences below ~0.01 on that row are noise.

In [ ]:
del optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()

model.eval()
val_loader = DataLoader(val_ds, batch_size=64, collate_fn=collate)
probs, trues = [], []
t0 = time.time()
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids=batch['input_ids'],
                           attention_mask=batch['attention_mask']).logits
        probs.append(torch.sigmoid(logits.float()).cpu())
        trues.append(batch['labels'].cpu())
val_probs = torch.cat(probs).numpy()
val_true = torch.cat(trues).numpy()
print(f'validated {len(val_probs):,} comments in {time.time() - t0:.1f}s\n')

aucs = {}
for i, label in enumerate(LABELS):
    aucs[label] = roc_auc_score(val_true[:, i], val_probs[:, i])
    delta = aucs[label] - BASELINE_VAL_AUC[label]
    flag = '+' if delta >= 0 else ''
    print(f'{label:14s} val AUC = {aucs[label]:.4f}  (baseline {BASELINE_VAL_AUC[label]:.4f}, {flag}{delta:.4f})')

mean_auc = float(np.mean(list(aucs.values())))
base_mean = float(np.mean(list(BASELINE_VAL_AUC.values())))
print(f'\nMEAN val AUC: DistilBERT {mean_auc:.4f}  vs  TF-IDF baseline {base_mean:.4f}')

In [ ]:
x = np.arange(len(LABELS))
w = 0.38
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - w / 2, [BASELINE_VAL_AUC[l] for l in LABELS], w,
            label='TF-IDF + LogReg', color=C_BASE)
b2 = ax.bar(x + w / 2, [aucs[l] for l in LABELS], w,
            label='DistilBERT (fine-tuned)', color=C_BERT)
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.001,
                f'{b.get_height():.3f}', ha='center', fontsize=9)
ymin = min(0.90, *BASELINE_VAL_AUC.values(), *aucs.values()) - 0.005
ax.set_ylim(ymin, 1.0)
ax.set_xticks(x)
ax.set_xticklabels([PRETTY[l] for l in LABELS], fontsize=11)
ax.set_title('Per-class validation AUC: baseline vs fine-tuned DistilBERT')
ax.legend(loc='lower right')
fig.savefig(FIG_DIR / '09_transformer_vs_baseline_auc.png', dpi=140, bbox_inches='tight')
plt.show()

## 6. Predict the test set -> submission #2

Trained on 80% only (we deliberately skip the full-data refit to keep this notebook under an hour). Production would retrain on 100% - noted in the takeaways.

In [ ]:
test_loader = DataLoader(test_ds, batch_size=64, collate_fn=collate)
test_probs = []
t0 = time.time()
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids=batch['input_ids'],
                           attention_mask=batch['attention_mask']).logits
        test_probs.append(torch.sigmoid(logits.float()).cpu())
test_probs = torch.cat(test_probs).numpy()
print(f'predicted {len(test_probs):,} test comments in {(time.time() - t0) / 60:.1f} min')

sub = pd.DataFrame({'id': test['id']})
for i, label in enumerate(LABELS):
    sub[label] = test_probs[:, i]
SUB_PATH = SUB_DIR / 'submission_02_distilbert.csv'
sub.to_csv(SUB_PATH, index=False)
print(f'wrote {SUB_PATH}  ({SUB_PATH.stat().st_size / 1024 / 1024:.1f} MB)')
sub.head()

In [ ]:
r = subprocess.run(
    ['python', '-m', 'kaggle', 'competitions', 'submit',
     '-c', 'jigsaw-toxic-comment-classification-challenge',
     '-f', str(SUB_PATH),
     '-m', 'nb04: distilbert-base-uncased 1ep 80pct max_len192 fp16'],
    capture_output=True, text=True, timeout=300)
print(r.stdout[-800:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-800:])

In [ ]:
# Scoring takes ~1 min. Re-run this cell if the status says 'pending'.
r = subprocess.run(
    ['python', '-m', 'kaggle', 'competitions', 'submissions',
     '-c', 'jigsaw-toxic-comment-classification-challenge'],
    capture_output=True, text=True, timeout=120)
print(r.stdout[:1200])

## 7. Takeaways & scaling path

1. **Representation learning beat feature engineering** - pretrained English knowledge + 1 epoch of adaptation vs a bag-of-words baseline. The exact gain depends on the class; the biggest wins usually land on context-dependent cases (insults wrapped in polite sentences).
2. **The fine-tuning recipe is small and standard**: fresh `Linear(768, 6)` head, BCE loss, AdamW 2e-5, warmup, 1 epoch, fp16. Everything else in this notebook is plumbing you now know.
3. **We left performance on the table on purpose** (80% train, 1 epoch, DistilBERT). To push toward 0.98+: refit on 100% of data, 2-3 epochs, `roberta-base` / `deberta-v3`, and ensemble the linear + transformer predictions. The 2018 winners (0.986) stacked many models - not clever tricks, just more of the same, ensembled.

**Possible notebook 05:** error analysis - pull the comments where the two models disagree and read *why*; it is the fastest way to build intuition for what the transformer actually learned.